In [32]:
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.workflow.pipeline import Pipeline

pipeline_session = PipelineSession()
region = pipeline_session.boto_region_name
role = sagemaker.get_execution_role()
bucket = pipeline_session.default_bucket()
prefix = "emi-y-mora/pipeline-byoc"

preprocess_image_uri = "780191826160.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-preprocess:latest"
train_image_uri = "780191826160.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-train:latest"
serve_image_uri = "780191826160.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-serve:latest"

processing_instance_type = ParameterString(
    name="ProcessingInstanceType",
    default_value="ml.m5.large"
)

training_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.large"
)

transform_instance_type = ParameterString(
    name="TransformInstanceType",
    default_value="ml.m5.large"
)

model_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval"
)

rmse_threshold = ParameterFloat(
    name="RMSEThreshold",
    default_value=1.0
)

print("role:", role)
print("bucket:", bucket)
print("region:", region)
print("prefix:", prefix)

role: arn:aws:iam::780191826160:role/SageMakerStudioExecutionRole2026
bucket: sagemaker-us-east-1-780191826160
region: us-east-1
prefix: emi-y-mora/pipeline-byoc


In [33]:
from pathlib import Path

from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput, TransformInput
from sagemaker.model import Model
from sagemaker.transformer import Transformer
from sagemaker.model_metrics import MetricsSource, ModelMetrics

from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TransformStep
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.functions import JsonGet, Join
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.fail_step import FailStep

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("repo_root:", repo_root)

repo_root: /home/sagemaker-user/Emi-y-Mora


In [14]:
preprocess_processor = ScriptProcessor(
    image_uri=preprocess_image_uri,
    command=["python"],
    role=role,
    instance_count=1,
    instance_type=processing_instance_type,
    sagemaker_session=pipeline_session,
    base_job_name="emi-y-mora-preprocess",
)

train_estimator = Estimator(
    image_uri=train_image_uri,
    role=role,
    instance_count=1,
    instance_type=training_instance_type,
    output_path=f"s3://{bucket}/{prefix}/training-output",
    sagemaker_session=pipeline_session,
    base_job_name="emi-y-mora-train",
)

In [34]:
step_process = ProcessingStep(
    name="Preprocess",
    processor=preprocess_processor,
    inputs=[
        ProcessingInput(
            source=str(repo_root / "data" / "raw"),
            destination="/opt/ml/processing/input",
            input_name="raw",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/output/train",
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/output/validation",
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/output/test",
        ),
    ],
    code=str(repo_root / "src" / "prep.py"),
)

In [35]:
step_train = TrainingStep(
    name="Train",
    estimator=train_estimator,
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    },
)

In [36]:
evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

step_evaluate = ProcessingStep(
    name="Evaluate",
    processor=preprocess_processor,
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/input/model",
            input_name="model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/input/test",
            input_name="test",
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/output/evaluation",
        )
    ],
    code=str(repo_root / "src" / "evaluate.py"),
    property_files=[evaluation_report],
)

In [37]:
serve_model = Model(
    image_uri=serve_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=pipeline_session,
    name="emi-y-mora-serve-model",
)

step_create_model = ModelStep(
    name="CreateModel",
    step_args=serve_model.create(
        instance_type=transform_instance_type,
    ),
)

In [38]:
transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_count=1,
    instance_type=transform_instance_type,
    output_path=f"s3://{bucket}/{prefix}/batch-output",
    sagemaker_session=pipeline_session,
)

step_transform = TransformStep(
    name="BatchTransform",
    transformer=transformer,
    inputs=TransformInput(
        data=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
        content_type="text/csv",
        split_type="Line",
    ),
)

In [39]:
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/",
            values=[
                step_evaluate.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
                "evaluation.json",
            ],
        ),
        content_type="application/json",
    )
)

step_register = RegisterModel(
    name="RegisterModel",
    estimator=train_estimator,
    image_uri=serve_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    model_package_group_name="EmiYMoraModelPackageGroup",
    approval_status=model_approval_status,
    content_types=["text/csv", "application/json"],
    response_types=["application/json"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_metrics=model_metrics,
)

In [40]:
rmse_value = JsonGet(
    step_name=step_evaluate.name,
    property_file=evaluation_report,
    json_path="regression_metrics.rmse.value",
)

step_fail = FailStep(
    name="FailOnHighRMSE",
    error_message=Join(
        on="",
        values=[
            "RMSE above threshold. Threshold=",
            rmse_threshold,
        ],
    ),
)

step_condition = ConditionStep(
    name="CheckRMSE",
    conditions=[
        ConditionLessThanOrEqualTo(
            left=rmse_value,
            right=rmse_threshold,
        )
    ],
    if_steps=[step_create_model, step_transform, step_register],
    else_steps=[step_fail],
)

In [41]:
pipeline = Pipeline(
    name="EmiYMoraPipelineBYOC",
    parameters=[
        processing_instance_type,
        training_instance_type,
        transform_instance_type,
        model_approval_status,
        rmse_threshold,
    ],
    steps=[
        step_process,
        step_train,
        step_evaluate,
        step_condition,
    ],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(role_arn=role)
execution = pipeline.start()
print("Pipeline execution ARN:", execution.arn)

Pipeline execution ARN: arn:aws:sagemaker:us-east-1:780191826160:pipeline/EmiYMoraPipelineBYOC/execution/ayesgj2s902i


In [42]:
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:780191826160:pipeline/EmiYMoraPipelineBYOC',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:780191826160:pipeline/EmiYMoraPipelineBYOC/execution/ayesgj2s902i',
 'PipelineExecutionDisplayName': 'execution-1774318457333',
 'PipelineExecutionStatus': 'Executing',
 'PipelineExperimentConfig': {'ExperimentName': 'EmiYMoraPipelineBYOC',
  'TrialName': 'ayesgj2s902i'},
 'CreationTime': datetime.datetime(2026, 3, 24, 2, 14, 17, 231000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 3, 24, 2, 14, 17, 231000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:780191826160:user-profile/d-5k4j2nbjajkd/datascientist',
  'UserProfileName': 'datascientist',
  'DomainId': 'd-5k4j2nbjajkd',
  'IamIdentity': {'Arn': 'arn:aws:sts::780191826160:assumed-role/SageMakerStudioExecutionRole2026/SageMaker',
   'PrincipalId': 'AROA3LJYOQDYIZIAG3ONL:SageMaker'}},
 'LastModifiedBy': {'UserProfileArn': 'arn:aws:sagema

In [43]:
execution.list_steps()

[{'StepName': 'Preprocess',
  'StartTime': datetime.datetime(2026, 3, 24, 2, 14, 17, 998000, tzinfo=tzlocal()),
  'StepStatus': 'Executing',
  'Metadata': {'ProcessingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:780191826160:processing-job/pipelines-ayesgj2s902i-Preprocess-uGSTm0oGbc'}},
  'AttemptCount': 1}]

In [45]:
steps = execution.list_steps()
for s in steps:
    print("STEP:", s["StepName"])
    print("  STATUS:", s["StepStatus"])
    print("  START :", s.get("StartTime"))
    print("  END   :", s.get("EndTime"))
    print("  FAIL  :", s.get("FailureReason"))
    print("-" * 60)

STEP: Preprocess
  STATUS: Failed
  START : 2026-03-24 02:14:17.998000+00:00
  END   : 2026-03-24 02:16:52.099000+00:00
  FAIL  : ClientError: AlgorithmError: , exit code: 1
------------------------------------------------------------


In [46]:
steps = execution.list_steps()
pre_arn = steps[0]["Metadata"]["ProcessingJob"]["Arn"]
pre_job_name = pre_arn.split("/")[-1]
print(pre_job_name)

pipelines-ayesgj2s902i-Preprocess-uGSTm0oGbc


In [47]:
import boto3

sm = boto3.client("sagemaker")

desc = sm.describe_processing_job(ProcessingJobName=pre_job_name)
print("FailureReason:", desc.get("FailureReason"))
print("ExitMessage:", desc.get("ExitMessage"))
print("AppSpecification:", desc.get("AppSpecification"))

FailureReason: AlgorithmError: , exit code: 1
ExitMessage: None
AppSpecification: {'ImageUri': '780191826160.dkr.ecr.us-east-1.amazonaws.com/emi-y-mora-preprocess:latest', 'ContainerEntrypoint': ['python', '/opt/ml/processing/input/code/prep.py']}


In [48]:
logs = boto3.client("logs")
group = "/aws/sagemaker/ProcessingJobs"

streams = logs.describe_log_streams(
    logGroupName=group,
    logStreamNamePrefix=pre_job_name,
    orderBy="LogStreamName",
    descending=True,
)

for s in streams["logStreams"][:5]:
    print(s["logStreamName"])

pipelines-ayesgj2s902i-Preprocess-uGSTm0oGbc/algo-1-1774318501


In [49]:
stream_name = streams["logStreams"][0]["logStreamName"]

events = logs.get_log_events(
    logGroupName=group,
    logStreamName=stream_name,
    startFromHead=True,
)

for e in events["events"][-80:]:
    print(e["message"])

Traceback (most recent call last):
  File "/opt/ml/processing/input/code/prep.py", line 32, in <module>
    from src.utils.logging_config import setup_logger
ModuleNotFoundError: No module named 'src'
